In [1]:
import sys, os, glob, shutil

print("=" * 70)
print("STEP 1: PLATFORM DETECTION & PERMANENT CACHE")
print("=" * 70)

PERSISTENT_DIR = "/kaggle/working"
OUTPUT_BASE = "/kaggle/working"

# --- MOUNT PERMANENT MODEL CACHE ---
MOUNTED_CACHE = "/kaggle/input/datasets/bahaeeddineessadki/marker-model-cache"
LOCAL_CACHE = os.path.join(PERSISTENT_DIR, "marker_models_cache")

# Check if cache is already populated from a previous run this session
CACHE_MARKER = os.path.join(LOCAL_CACHE, ".cache_ready")
MODELS_CACHED = False

if os.path.exists(CACHE_MARKER):
    print(f"\n✅ Local cache already populated — skipping copy.")
    MODELS_CACHED = True
elif os.path.exists(MOUNTED_CACHE):
    mounted_items = os.listdir(MOUNTED_CACHE)
    if len(mounted_items) == 0:
        print(f"\n⚠️  Dataset mounted but empty! Models will download on first use.")
    else:
        print(f"\n📦 Found permanent dataset: {MOUNTED_CACHE}")

        # Copy (more reliable than symlink for HF cache)
        if os.path.islink(LOCAL_CACHE):
            os.unlink(LOCAL_CACHE)
        elif os.path.isdir(LOCAL_CACHE):
            shutil.rmtree(LOCAL_CACHE)

        print("   Copying to local cache folder... (takes ~30 sec for 3GB)")
        shutil.copytree(MOUNTED_CACHE, LOCAL_CACHE)
        print(f"✅ Copied to: {LOCAL_CACHE}")
        print("   Models ready — NO DOWNLOAD NEEDED!")

        # Write marker so subsequent runs skip the copy
        with open(CACHE_MARKER, "w") as f:
            f.write("ready")
        MODELS_CACHED = True
else:
    print(f"\n⚠️  Dataset NOT mounted!")
    print("   Click 'Add Data' → 'Your Datasets' → 'marker-model-cache'")
    print("   Then re-run this cell.")
    os.makedirs(LOCAL_CACHE, exist_ok=True)

# Set env vars
os.environ["HF_HOME"] = LOCAL_CACHE
os.environ["TRANSFORMERS_CACHE"] = os.path.join(LOCAL_CACHE, "transformers")
os.environ["TORCH_HOME"] = os.path.join(LOCAL_CACHE, "torch")
os.environ["XDG_CACHE_HOME"] = LOCAL_CACHE

# GPU Check
try:
    import torch
    print(f"\n🔥 PyTorch: {torch.__version__}")
    print(f"📟 CUDA: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")
        print(f"💾 VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
except ImportError:
    print("\n⚠️  PyTorch not installed yet.")

print("\n✅ Setup complete.")

STEP 1: PLATFORM DETECTION & PERMANENT CACHE

📦 Found permanent dataset: /kaggle/input/datasets/bahaeeddineessadki/marker-model-cache
   Copying to local cache folder... (takes ~30 sec for 3GB)
✅ Copied to: /kaggle/working/marker_models_cache
   Models ready — NO DOWNLOAD NEEDED!

🔥 PyTorch: 2.10.0+cu128
📟 CUDA: True
🎮 GPU: Tesla T4
💾 VRAM: 15.6 GB

✅ Setup complete.


In [2]:
import os

print("\n" + "=" * 70)
print("STEP 2: INSTALL MARKER")
print("=" * 70)

# Define CACHE_DIR if missing (e.g., after kernel restart)
try:
    CACHE_DIR
except NameError:
    CACHE_DIR = "/kaggle/working/marker_models_cache"
    os.makedirs(CACHE_DIR, exist_ok=True)
    print(f"⚠️  CACHE_DIR not found. Using default: {CACHE_DIR}")

os.environ["HF_HOME"] = CACHE_DIR
#os.environ["TRANSFORMERS_CACHE"] = os.path.join(CACHE_DIR, "transformers") #could be deleted based on claude

try:
    import marker
    print(f"\u2705 Marker already installed (v{marker.__version__})")
except ImportError:
    print("\u23f3 Installing marker-pdf...")
    !pip install marker-pdf -q
    print("\u2705 Marker installed.")

!which marker_single
print("\n\u2705 Ready.")


STEP 2: INSTALL MARKER
⚠️  CACHE_DIR not found. Using default: /kaggle/working/marker_models_cache
/usr/local/bin/marker_single

✅ Marker installed.


In [3]:
print("\n" + "=" * 70)
print("STEP 3: CACHE STATUS")
print("=" * 70)

# Re-check cache
cache_size_gb = 0.0
if os.path.exists(CACHE_DIR):
    cache_size_gb = sum(
        os.path.getsize(os.path.join(dp, f))
        for dp, dn, filenames in os.walk(CACHE_DIR)
        for f in filenames
    ) / (1024 ** 3)

if cache_size_gb > 1.0:
    print(f"\n✅ Found {cache_size_gb:.1f} GB cached models.")
    print("   marker_single will load from cache instantly.")
else:
    print(f"\n⚠️  Cache empty ({cache_size_gb:.1f} GB).")
    print("   Models will auto-download during extraction (Cell 5).")
    print("   This is normal on first run — takes ~3-5 minutes.")

print(f"\n📂 Cache location: {CACHE_DIR}")
print("✅ Ready for PDF processing.")


STEP 3: CACHE STATUS

✅ Found 3.3 GB cached models.
   marker_single will load from cache instantly.

📂 Cache location: /kaggle/working/marker_models_cache
✅ Ready for PDF processing.


# <span style="font-size: 50px;">🛑 🚨 ⚠️ **CRITICAL WARNING** ⚠️ 🚨 🛑</span>

<div style="font-size: 24px; color: #cc0000; border-left: 8px solid #cc0000; padding: 15px; background-color: #fff5f5; margin-top: 10px;">

**STOP BEFORE RUNNING:** 

This cell or notebook contains actions that require your absolute attention. 
* Do not proceed without checking THE SETTINGS BLOCK BELOW .
* Proceeding may overwrite existing data files.
</div>


In [ ]:
import json, os, threading, sys, time

# ============================================================
# CONFIG UI: Launches a FastAPI server + Cloudflare tunnel
# ============================================================

CONFIG_PATH = "/kaggle/working/config.json"
DEFAULTS = {
    "mode": "single",
    "split_at_page": 12,
    "batch_input_dir": "/kaggle/working/pdf_batch",
    "drive_link": "",
    "drive_folder_link": "",
    "zip_name": "output",
    "add_page_markers": False,
    "github_token": "",
}

def load_config():
    if os.path.exists(CONFIG_PATH):
        with open(CONFIG_PATH) as f:
            saved = json.load(f)
        cfg = DEFAULTS.copy()
        cfg.update({k: v for k, v in saved.items() if k in DEFAULTS})
        if cfg.get("mode") in ("auto", "default"):
            cfg["mode"] = "single"
        return cfg
    return dict(DEFAULTS)

# Install deps if missing
try:
    import fastapi
except ImportError:
    print("Installing fastapi + uvicorn...")
    !pip install fastapi uvicorn -q
    import fastapi

# Download cloudflared binary (no API key needed)
import urllib.request, stat
CLOUDFLARED_PATH = "/kaggle/working/cloudflared"
if not os.path.exists(CLOUDFLARED_PATH):
    print("Downloading cloudflared...")
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        CLOUDFLARED_PATH
    )
    os.chmod(CLOUDFLARED_PATH, stat.S_IRWXU | stat.S_IRGRP | stat.S_IXGRP | stat.S_IROTH | stat.S_IXOTH)
    print("cloudflared downloaded")

from fastapi import FastAPI, Request
from fastapi.responses import HTMLResponse, FileResponse
from fastapi.middleware.cors import CORSMiddleware
from pathlib import Path

app = FastAPI(title="Marker PDF Converter")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

DOWNLOAD_DIR = "/kaggle/working"

@app.get("/", response_class=HTMLResponse)
async def index():
    return HTML_STR

@app.get("/api/load")
async def load():
    return load_config()

@app.post("/api/save")
async def save(request: Request):
    body = await request.json()
    if "data" in body and isinstance(body["data"], list):
        values = body["data"][0]
    else:
        values = body
    cfg = DEFAULTS.copy()
    cfg.update({k: v for k, v in values.items() if k in DEFAULTS})
    os.makedirs(os.path.dirname(CONFIG_PATH) or ".", exist_ok=True)
    with open(CONFIG_PATH, "w") as f:
        json.dump(cfg, f, indent=2)
    if cfg.get("github_token"):
        os.environ["GITHUB_TOKEN"] = cfg["github_token"]
    lines = "\n".join(f"  {k} = {v!r}" for k, v in cfg.items())
    return {"status": "ok", "message": f"Config saved.\n\n{lines}"}

@app.post("/api/reset")
async def reset():
    return DEFAULTS

def human_size(path):
    b = os.path.getsize(path)
    for unit in ("B", "KB", "MB", "GB"):
        if b < 1024:
            return f"{b:.0f} {unit}"
        b /= 1024
    return f"{b:.1f} GB"

@app.get("/api/files")
async def list_files():
    try:
        files = []
        for p in sorted(Path(DOWNLOAD_DIR).iterdir(), key=lambda x: x.stat().st_mtime, reverse=True):
            if p.suffix.lower() in (".zip", ".md", ".txt", ".pdf", ".json") and p.is_file():
                files.append({"name": p.name, "size": human_size(p)})
        return {"files": files}
    except Exception as e:
        return {"files": [], "error": str(e)}

@app.get("/api/download/{filename:path}")
async def download_file(filename: str):
    filepath = os.path.join(DOWNLOAD_DIR, filename)
    if not os.path.isfile(filepath):
        return HTMLResponse("File not found", status_code=404)
    return FileResponse(filepath, filename=filename)

HTML_STR = r"""<!-- ============================================================
     Marker PDF Converter — Config UI
     Edit this file with any HTML/CSS/JS tool to customize the look.
     The Python backend reads inputs by their 'data-testid' attributes.
     ============================================================ -->
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1.0" />
  <title>Marker PDF Converter</title>
  <link rel="preconnect" href="https://fonts.googleapis.com">
  <link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
  <link href="https://fonts.googleapis.com/css2?family=Fraunces:opsz,wght@9..144,500;9..144,600;9..144,800&family=Inter:wght@400;500;600;700&family=JetBrains+Mono:wght@400;500;700&display=swap" rel="stylesheet">
  <style>
    :root {
      --bg: #eae6db;
      --bg-grain: rgba(30,28,20,.035);
      --card-bg: #fffdf8;
      --card-shadow: 0 1px 2px rgba(30,25,15,.04), 0 12px 32px -12px rgba(30,25,15,.18);
      --ink: #1e2233;
      --ink-soft: #62667a;
      --ink-hint: #93968f;
      --border: #ddd6c4;
      --input-bg: #fffdf8;
      --status-bg: #f4f0e4;
      --status-border: #e3ddc9;
      --status-success-bg: #eef8ee;
      --status-success-border: #3f9142;
      --pill-bg: #fffdf8;
      --pill-active-bg: #fff3c4;
      --pill-active-border: #cc9a06;
      --btn-secondary-bg: #fffdf8;
      --btn-secondary-text: #1e2233;
      --btn-secondary-hover: #f4f0e4;
      --accent: #ffd23f;
      --accent-ink: #3d2fc4;
      --accent-ink-hover: #2f22a0;
      --tape: #f3d9536b;
      --mark-blend: multiply;
      --mark-opacity: .55;
      --scrollbar: #d8d0ba;
      --stage-done: #34a853;
      --stage-current: #3d2fc4;
      --stage-pending: #d8d0ba;
      --progress-track: #e3ddc9;
      --progress-fill: #3d2fc4;
    }
    body.dark {
      --bg: #111421;
      --bg-grain: rgba(255,255,255,.02);
      --card-bg: #171b2b;
      --card-shadow: 0 1px 0 rgba(255,255,255,.03) inset, 0 20px 48px -16px rgba(0,0,0,.65);
      --ink: #eae7dd;
      --ink-soft: #9a9db3;
      --ink-hint: #6c6f85;
      --border: #2a2e44;
      --input-bg: #0f1220;
      --status-bg: #0f1220;
      --status-border: #262a40;
      --status-success-bg: #0e2318;
      --status-success-border: #34a853;
      --pill-bg: transparent;
      --pill-active-bg: #2a2410;
      --pill-active-border: #ffd23f;
      --btn-secondary-bg: transparent;
      --btn-secondary-text: #eae7dd;
      --btn-secondary-hover: #1d2136;
      --accent: #ffd23f;
      --accent-ink: #8a7bff;
      --accent-ink-hover: #a294ff;
      --tape: #ffd23f2e;
      --mark-blend: screen;
      --mark-opacity: .22;
      --scrollbar: #2a2e44;
      --stage-done: #34a853;
      --stage-current: #8a7bff;
      --stage-pending: #3a3e55;
      --progress-track: #2a2e44;
      --progress-fill: #8a7bff;
    }
    * { box-sizing: border-box; margin: 0; padding: 0; }
    html { scrollbar-color: var(--scrollbar) transparent; }
    body {
      font-family: 'Inter', -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
      background:
        radial-gradient(ellipse 900px 500px at 15% -10%, var(--bg-grain), transparent),
        var(--bg);
      color: var(--ink);
      display: flex; justify-content: center; padding: 56px 20px;
      transition: background .3s, color .3s;
      min-height: 100vh;
    }

    .stage {
      max-width: 720px; width: 100%;
    }

    /* ---------- Step rail reflects current view ---------- */
    .step-rail {
      display: flex; gap: 6px; margin-bottom: 14px; padding: 0 4px;
    }
    .step-rail .step {
      flex: 1; display: flex; align-items: baseline; gap: 6px;
      font-family: 'JetBrains Mono', monospace; font-size: 11px; letter-spacing: .04em;
      color: var(--ink-hint); padding-bottom: 8px; border-bottom: 2px solid var(--border);
      transition: color .3s, border-color .3s;
    }
    .step-rail .step b { color: var(--ink-soft); font-weight: 700; }
    .step-rail .step.is-current { color: var(--accent-ink); border-bottom-color: var(--accent); }
    .step-rail .step.is-current b { color: var(--accent-ink); }
    .step-rail .step.is-done { color: var(--stage-done); border-bottom-color: var(--stage-done); }
    .step-rail .step.is-done b { color: var(--stage-done); }

    /* ---------- Card ---------- */
    .card {
      background: var(--card-bg); border-radius: 14px;
      box-shadow: var(--card-shadow);
      border: 1px solid var(--border);
      padding: 36px 34px 32px; width: 100%;
      position: relative; overflow: hidden;
      transition: background .3s, box-shadow .3s, border-color .3s;
    }
    .card::before {
      content: "";
      position: absolute; top: -14px; left: 34px;
      width: 74px; height: 26px;
      background: var(--tape);
      border: 1px solid rgba(0,0,0,.04);
      transform: rotate(-3.5deg);
      box-shadow: 0 2px 4px rgba(0,0,0,.08);
      pointer-events: none;
    }

    .header { display: flex; justify-content: space-between; align-items: flex-start; gap: 16px; }
    .header-text h1 {
      font-family: 'Fraunces', serif; font-weight: 600; font-size: 26px;
      color: var(--ink); letter-spacing: -0.01em; line-height: 1.15;
    }
    .header-text h1 .mark {
      position: relative; display: inline-block;
    }
    .header-text h1 .mark::before {
      content: "";
      position: absolute; left: -3px; right: -4px; bottom: 1px; height: 13px;
      background: var(--accent);
      opacity: var(--mark-opacity); mix-blend-mode: var(--mark-blend);
      border-radius: 3px 8px 5px 9px / 7px 4px 8px 3px;
      transform: rotate(-1deg);
      z-index: 0;
    }
    .header-text h1 .mark span { position: relative; z-index: 1; }
    .header-text p.sub { color: var(--ink-soft); font-size: 13.5px; margin-top: 6px; }

    .theme-toggle {
      flex-shrink: 0;
      background: var(--pill-bg); border: 1px solid var(--border); border-radius: 20px;
      padding: 6px; cursor: pointer; font-size: 15px; line-height: 1;
      color: var(--ink); transition: background .15s, border-color .15s;
      width: 52px; height: 30px; display: flex; align-items: center;
      justify-content: flex-start;
    }
    .theme-toggle .knob {
      width: 20px; height: 20px; border-radius: 50%;
      background: var(--accent); display: flex; align-items: center; justify-content: center;
      font-size: 11px; transition: transform .25s cubic-bezier(.4,0,.2,1);
    }
    body.dark .theme-toggle .knob { transform: translateX(20px); }
    .theme-toggle:hover { border-color: var(--accent-ink); }

    /* ---------- Views: only one visible at a time ---------- */
    .view-container { position: relative; min-height: 280px; }
    .view {
      transition: opacity .4s ease, transform .4s ease;
      opacity: 0; transform: translateY(10px);
      display: none;
    }
    .view.active {
      opacity: 1; transform: translateY(0);
      display: block;
    }

    /* ---------- View 1: Settings ---------- */
    .field { margin-bottom: 22px; }
    label {
      display: block; font-weight: 600; font-size: 12.5px; color: var(--ink);
      margin-bottom: 6px; letter-spacing: .01em;
    }
    .hint { font-size: 11.5px; color: var(--ink-hint); margin-top: 5px; }
    .row { display: flex; gap: 16px; }
    .row > * { flex: 1; min-width: 0; }

    input[type="text"], input[type="number"], input[type="password"] {
      width: 100%; padding: 10px 13px;
      border: 1px solid var(--border); border-radius: 9px;
      font-size: 14px; font-family: inherit; outline: none;
      transition: border-color .15s, background .3s, box-shadow .15s;
      background: var(--input-bg); color: var(--ink);
    }
    input::placeholder { color: var(--ink-hint); }
    input:focus-visible, .radio-group label:has(input:focus-visible) {
      border-color: var(--accent-ink);
      box-shadow: 0 0 0 3px color-mix(in srgb, var(--accent-ink) 18%, transparent);
    }
    input:-webkit-autofill,
    input:-webkit-autofill:focus {
      -webkit-box-shadow: 0 0 0 1000px var(--input-bg) inset !important;
      -webkit-text-fill-color: var(--ink) !important;
    }

    .radio-group { display: flex; gap: 10px; flex-wrap: wrap; }
    .radio-group label {
      display: flex; align-items: center; gap: 5px;
      font-weight: 500; font-size: 13px; cursor: pointer;
      padding: 7px 16px; border: 1px solid var(--border); border-radius: 8px;
      transition: transform .15s, border-color .15s, background .15s;
      background: var(--pill-bg); color: var(--ink-soft);
    }
    .radio-group label:nth-child(1) { transform: rotate(-0.6deg); }
    .radio-group label:nth-child(2) { transform: rotate(0.4deg); }
    .radio-group label:nth-child(3) { transform: rotate(-0.3deg); }
    .radio-group input:checked + span { color: var(--accent-ink); font-weight: 700; }
    .radio-group label:has(input:checked) {
      border-color: var(--pill-active-border); background: var(--pill-active-bg);
      transform: rotate(0deg) translateY(-1px);
    }
    .radio-group label:hover { border-color: var(--ink-hint); }
    .radio-group input { display: none; }

    .checkbox-row {
      display: flex; align-items: center; gap: 9px; margin-bottom: 22px;
    }
    .checkbox-row input { width: 18px; height: 18px; accent-color: var(--accent-ink); cursor: pointer; }
    .checkbox-row label { margin-bottom: 0; font-weight: 500; cursor: pointer; }

    .actions { display: flex; gap: 12px; margin-top: 6px; }
    .btn-primary {
      flex: 2; padding: 12px; border: none; border-radius: 9px;
      background: var(--accent-ink); color: #fff; font-weight: 600; font-size: 14.5px;
      cursor: pointer; transition: background .15s, transform .1s;
      font-family: inherit;
    }
    .btn-primary:hover { background: var(--accent-ink-hover); }
    .btn-primary:active { transform: translateY(1px); }
    .btn-primary:disabled { opacity: .6; cursor: default; }
    .btn-secondary {
      flex: 1; padding: 12px; border: 1px solid var(--border); border-radius: 9px;
      background: var(--btn-secondary-bg); color: var(--btn-secondary-text);
      font-weight: 500; font-size: 14.5px; cursor: pointer; transition: background .15s;
      font-family: inherit;
    }
    .btn-secondary:hover { background: var(--btn-secondary-hover); }

    #status {
      margin-top: 18px; padding: 13px 14px; border-radius: 9px;
      font-family: 'JetBrains Mono', monospace; font-size: 12.5px; background: var(--status-bg);
      border: 1px dashed var(--status-border); min-height: 54px; white-space: pre-wrap;
      color: var(--ink-soft); transition: background .3s, border-color .3s, color .3s;
    }
    #status.success { border-style: solid; border-color: var(--status-success-border); background: var(--status-success-bg); color: var(--ink); }
    #status::before { content: "$ "; color: var(--ink-hint); }
    #status.success::before { content: "✓ "; color: var(--status-success-border); }

    /* ---------- Views 2-4: Progress / Marker Detail / Downloads ---------- */

    .mode-badge {
      display: inline-block;
      font-family: 'JetBrains Mono', monospace; font-size: 11px; font-weight: 500;
      padding: 3px 10px; border-radius: 5px;
      background: var(--pill-active-bg); border: 1px solid var(--pill-active-border);
      color: var(--accent-ink); margin-bottom: 14px;
    }

    .main-bar-track {
      width: 100%; height: 18px;
      background: var(--progress-track); border-radius: 9px;
      overflow: hidden; margin-bottom: 8px;
    }
    .main-bar-fill {
      height: 100%; width: 0%;
      background: var(--progress-fill); border-radius: 9px;
      transition: width .6s cubic-bezier(.4,0,.2,1);
    }
    .main-bar-label {
      display: flex; justify-content: space-between;
      font-family: 'JetBrains Mono', monospace; font-size: 12px;
      color: var(--ink-soft); margin-bottom: 16px;
    }

    .step-title {
      font-size: 16px; font-weight: 600; color: var(--ink);
      margin-bottom: 4px;
    }
    .step-elapsed {
      font-family: 'JetBrains Mono', monospace; font-size: 12px;
      color: var(--ink-hint); margin-bottom: 16px;
    }

    /* ---- Step checklist ---- */
    .checklist { list-style: none; margin-top: 14px; }
    .checklist li {
      display: flex; align-items: center; gap: 10px;
      padding: 6px 0; font-size: 13px; color: var(--ink-soft);
      border-bottom: 1px solid var(--border);
    }
    .checklist li:last-child { border-bottom: none; }
    .checklist .icon { width: 20px; text-align: center; font-size: 14px; }
    .checklist .icon.done { color: var(--stage-done); }
    .checklist .icon.current { color: var(--stage-current); }
    .checklist .icon.pending { color: var(--stage-pending); }
    .checklist .label { flex: 1; }
    .checklist .label.current { color: var(--ink); font-weight: 600; }
    .checklist .label.current::after {
      content: " ●"; animation: pulse 1.5s infinite;
    }
    @keyframes pulse {
      0%, 100% { opacity: 1; }
      50% { opacity: .3; }
    }

    /* ---- Marker stage pills ---- */
    .stages-row {
      display: flex; gap: 8px; flex-wrap: wrap; margin: 14px 0;
    }
    .stage-pill {
      display: flex; align-items: center; gap: 5px;
      padding: 6px 12px; border-radius: 7px;
      font-size: 12px; font-weight: 500;
      border: 1px solid var(--border);
      background: var(--pill-bg); color: var(--ink-hint);
      transition: all .3s;
    }
    .stage-pill.done { border-color: var(--stage-done); color: var(--stage-done); background: var(--status-success-bg); }
    .stage-pill.current { border-color: var(--stage-current); color: var(--ink); background: var(--pill-active-bg); animation: pill-pop .4s ease; }
    @keyframes pill-pop {
      0% { transform: scale(0.92); }
      100% { transform: scale(1); }
    }
    .stage-pill .sp-progress { font-family: 'JetBrains Mono', monospace; font-size: 11px; color: var(--ink-hint); }

    /* ---- Sub-progress bar for current stage ---- */
    .sub-bar-track {
      width: 100%; height: 6px;
      background: var(--progress-track); border-radius: 3px;
      overflow: hidden; margin: 8px 0 4px;
    }
    .sub-bar-fill {
      height: 100%; width: 0%;
      background: var(--accent); border-radius: 3px;
      transition: width .5s cubic-bezier(.4,0,.2,1);
    }
    .sub-bar-label {
      font-family: 'JetBrains Mono', monospace; font-size: 11px;
      color: var(--ink-hint); text-align: right;
    }

    /* ---- File name display ---- */
    .file-header {
      display: flex; align-items: center; gap: 8px;
      padding: 10px 12px; border-radius: 8px;
      background: var(--status-bg); margin-bottom: 14px;
    }
    .file-header .fn { font-weight: 600; font-size: 14px; flex: 1; overflow: hidden; text-overflow: ellipsis; white-space: nowrap; }
    .file-header .fn-eta { font-family: 'JetBrains Mono', monospace; font-size: 12px; color: var(--ink-hint); }

    /* ---- Per-file times ---- */
    .per-file-list { list-style: none; margin-top: 12px; }
    .per-file-list li {
      display: flex; align-items: center; gap: 8px;
      padding: 5px 0; font-size: 12.5px; color: var(--ink-soft);
      font-family: 'JetBrains Mono', monospace;
    }
    .per-file-list .pf-done { color: var(--stage-done); }
    .per-file-list .pf-active { color: var(--stage-current); font-weight: 600; }

    /* ---- Files list in Downloads view ---- */
    .downloads { margin-top: 4px; }
    .downloads h3 {
      font-family: 'JetBrains Mono', monospace; font-weight: 500; font-size: 12px;
      text-transform: uppercase; letter-spacing: .06em;
      color: var(--ink-soft); margin-bottom: 10px; display: flex; gap: 10px; align-items: center;
    }
    .downloads h3 button {
      background: none; border: 1px solid var(--border); border-radius: 6px;
      cursor: pointer; font-size: 11px; padding: 3px 9px; color: var(--ink-soft);
      font-family: inherit; letter-spacing: normal; text-transform: none;
      transition: background .15s;
    }
    .downloads h3 button:hover { background: var(--btn-secondary-hover); }
    .file-list { list-style: none; }
    .file-list li {
      display: flex; justify-content: space-between; align-items: center;
      padding: 9px 12px; border-radius: 8px; background: var(--status-bg);
      border: 1px solid var(--status-border);
      margin-bottom: 7px; font-size: 13px; color: var(--ink);
    }
    .file-list li span { font-family: 'JetBrains Mono', monospace; font-size: 12px; color: var(--ink-soft); }
    .file-list li a {
      background: var(--accent-ink); color: #fff; text-decoration: none;
      padding: 5px 13px; border-radius: 6px; font-size: 11.5px; font-weight: 600;
      letter-spacing: .02em; transition: background .15s;
    }
    .file-list li a:hover { background: var(--accent-ink-hover); }
    .file-list .no-files { color: var(--ink-hint); font-style: italic; padding: 8px 0; font-family: 'JetBrains Mono', monospace; font-size: 12px; }

    footer.note {
      text-align: center; margin-top: 18px; font-size: 11.5px; color: var(--ink-hint);
      font-family: 'JetBrains Mono', monospace;
    }

    @media (max-width: 560px) {
      .row { flex-direction: column; gap: 0; }
      .step-rail .step span.step-label { display: none; }
      .card { padding: 28px 20px 24px; }
      .stages-row { gap: 5px; }
      .stage-pill { font-size: 11px; padding: 4px 8px; }
    }
    @media (prefers-reduced-motion: reduce) {
      * { transition-duration: .001ms !important; animation-duration: .001ms !important; }
    }
  </style>
</head>
<body>
  <div class="stage">
    <nav class="step-rail" aria-hidden="true">
      <div class="step is-current" data-step="1"><b>01</b><span class="step-label">Configure</span></div>
      <div class="step" data-step="2"><b>02</b><span class="step-label">Prepare</span></div>
      <div class="step" data-step="3"><b>03</b><span class="step-label">Process</span></div>
      <div class="step" data-step="4"><b>04</b><span class="step-label">Download</span></div>
    </nav>

    <div class="card">
      <div class="header">
        <div class="header-text">
          <h1><span class="mark"><span>Marker</span></span> PDF Converter</h1>
          <p class="sub" id="header-sub">Configure settings and click Save &amp; Apply.</p>
        </div>
        <button class="theme-toggle" id="btn-theme" title="Toggle dark mode">
          <span class="knob">🌙</span>
        </button>
      </div>

      <div class="view-container">
        <!-- ================================================================ -->
        <!-- VIEW 1: SETTINGS                                                 -->
        <!-- ================================================================ -->
        <div id="view-settings" class="view active">
          <!-- Mode -->
          <div class="field" data-field="mode" style="margin-top: 24px;">
            <label>Processing Mode</label>
            <div class="radio-group">
              <label><input type="radio" name="mode" value="single" checked><span>Single</span></label>
              <label><input type="radio" name="mode" value="split"><span>Split</span></label>
              <label><input type="radio" name="mode" value="folder"><span>Folder</span></label>
            </div>
          </div>

          <div class="row" id="row-drive-links">
            <div class="field" data-field="drive_link" id="field-drive-link">
              <label>Google Drive File Link</label>
              <input type="text" id="drive_link" placeholder="https://drive.google.com/d/..." />
            </div>
            <div class="field" data-field="drive_folder_link" id="field-folder-link">
              <label>Google Drive Folder Link</label>
              <input type="text" id="drive_folder_link" placeholder="https://drive.google.com/drive/folders/..." />
            </div>
          </div>

          <div class="row">
            <div class="field" data-field="split_at_page" id="field-split-page">
              <label>Split at Page</label>
              <input type="number" id="split_at_page" value="12" min="0" max="200" />
              <div class="hint">0 = auto half (split mode only)</div>
            </div>
          </div>

          <div class="row">
            <div class="field" data-field="zip_name">
              <label>Output ZIP Name</label>
              <input type="text" id="zip_name" value="output" />
            </div>
            <div class="checkbox-row" data-field="add_page_markers">
              <input type="checkbox" id="add_page_markers" />
              <label for="add_page_markers">Add Page Markers</label>
            </div>
          </div>

          <div class="field" data-field="github_token" style="margin-top: 4px;">
            <label>GitHub Token</label>
            <input type="password" id="github_token" placeholder="ghp_... or left empty" />
            <div class="hint">For auto-push to GitHub (Cell 9). Leave empty to skip.</div>
          </div>

          <div class="actions">
            <button class="btn-primary" id="btn-save">Save &amp; Apply</button>
            <button class="btn-secondary" id="btn-reset">Reset to Defaults</button>
          </div>

          <div id="status">Ready.</div>
        </div>

        <!-- ================================================================ -->
        <!-- VIEW 2: PREPARING                                                -->
        <!-- ================================================================ -->
        <div id="view-preparing" class="view">
          <span class="mode-badge" id="prep-mode-badge">Single</span>

          <div class="main-bar-track">
            <div class="main-bar-fill" id="prep-bar"></div>
          </div>
          <div class="main-bar-label">
            <span id="prep-pct">0%</span>
            <span id="prep-elapsed">⏱ 0s</span>
          </div>

          <div class="step-title" id="prep-title">Config saved. Initializing...</div>

          <ul class="checklist" id="prep-checklist">
            <li><span class="icon pending">○</span><span class="label">Config saved</span></li>
            <li><span class="icon pending">○</span><span class="label">Downloading PDF</span></li>
            <li><span class="icon pending">○</span><span class="label">Starting Marker</span></li>
          </ul>
        </div>

        <!-- ================================================================ -->
        <!-- VIEW 3: MARKER DETAIL                                            -->
        <!-- ================================================================ -->
        <div id="view-marker" class="view">
          <span class="mode-badge" id="marker-mode-badge">Single</span>

          <div class="main-bar-track">
            <div class="main-bar-fill" id="marker-bar"></div>
          </div>
          <div class="main-bar-label">
            <span id="marker-pct">0%</span>
            <span id="marker-elapsed">⏱ 0s</span>
          </div>

          <div class="step-title" id="marker-title">Loading models...</div>

          <div class="file-header">
            <span class="fn" id="marker-filename">doc_20260721_123456.pdf</span>
            <span class="fn-eta" id="marker-eta">ETA: --</span>
          </div>

          <!-- Marker stages row -->
          <div class="stages-row" id="stages-row">
            <span class="stage-pill" data-stage="Layout"><span class="icon">○</span> Layout</span>
            <span class="stage-pill" data-stage="OCR"><span class="icon">○</span> OCR</span>
            <span class="stage-pill" data-stage="bboxes"><span class="icon">○</span> bboxes</span>
            <span class="stage-pill" data-stage="Text"><span class="icon">○</span> Text</span>
            <span class="stage-pill" data-stage="tables"><span class="icon">○</span> tables</span>
          </div>

          <!-- Sub-progress for current stage -->
          <div style="margin-top: 6px;">
            <div class="sub-bar-track">
              <div class="sub-bar-fill" id="stage-sub-bar"></div>
            </div>
            <div class="sub-bar-label" id="stage-sub-label">0/0</div>
          </div>

          <!-- Previous file times -->
          <ul class="per-file-list" id="per-file-list"></ul>
        </div>

        <!-- ================================================================ -->
        <!-- VIEW 4: DOWNLOADS                                                -->
        <!-- ================================================================ -->
        <div id="view-downloads" class="view">
          <span class="mode-badge" id="dl-mode-badge">Single</span>

          <div class="step-title" style="font-size: 20px; margin-bottom: 4px;">✅ Complete!</div>
          <div class="step-elapsed" id="dl-elapsed">Finished in 0s</div>

          <div class="main-bar-track" style="height: 12px;">
            <div class="main-bar-fill" id="dl-bar" style="width: 100%;"></div>
          </div>
          <div class="main-bar-label" style="margin-bottom: 20px;">
            <span id="dl-pct">100%</span>
            <span id="dl-status">Done</span>
          </div>

          <div class="downloads">
            <h3>
              Downloads
              <button id="btn-refresh-files">Refresh</button>
            </h3>
            <ul class="file-list" id="file-list">
              <li class="no-files">No output files yet.</li>
            </ul>
          </div>
        </div>
      </div>

    </div>

    <footer class="note">marker · pdf → markdown, split, and ship</footer>
  </div>

  <script>
    // ============================================================
    // Dark mode toggle
    // ============================================================
    function setTheme(dark) {
      document.body.classList.toggle("dark", dark);
      document.querySelector("#btn-theme .knob").textContent = dark ? "☀️" : "🌙";
      localStorage.setItem("dark-mode", dark);
    }
    if (localStorage.getItem("dark-mode") === "true") setTheme(true);

    document.getElementById("btn-theme").addEventListener("click", () => {
      setTheme(!document.body.classList.contains("dark"));
    });

    // ============================================================
    // View switching
    // ============================================================
    const VIEWS = ["settings", "preparing", "marker", "downloads"];

    function switchView(viewId) {
      document.querySelectorAll(".view").forEach(el => el.classList.remove("active"));
      const target = document.getElementById("view-" + viewId);
      if (target) target.classList.add("active");

      // Update step rail
      const idx = VIEWS.indexOf(viewId);
      document.querySelectorAll(".step-rail .step").forEach((el, i) => {
        el.classList.toggle("is-current", i === idx);
        el.classList.toggle("is-done", i < idx);
      });

      // Update header subtitle
      document.getElementById("header-sub").textContent =
        viewId === "settings" ? "Configure settings and click Save &amp; Apply." :
        viewId === "downloads" ? "Your files are ready." :
        "Processing in progress...";
    }

    // ============================================================
    // Status polling
    // ============================================================
    async function pollStatus() {
      try {
        const res = await fetch("/api/status");
        const data = await res.json();
        renderStatus(data);
      } catch (e) {
        // Server not reachable — stay on settings
      }
    }

    function renderStatus(data) {
      const view = data.view || "settings";
      const pct = Math.round((data.progress || 0) * 100);
      const elapsed = data.elapsed || 0;
      const mode = data.mode || "single";

      switchView(view);

      if (view === "preparing") {
        document.getElementById("prep-mode-badge").textContent = capitalize(mode) + " Mode";
        document.getElementById("prep-bar").style.width = pct + "%";
        document.getElementById("prep-pct").textContent = pct + "%";
        document.getElementById("prep-elapsed").textContent = "⏱ " + formatTime(elapsed);
        document.getElementById("prep-title").textContent = data.step || "";

        // Update checklist
        updateChecklist("prep-checklist", data);
      }

      if (view === "marker") {
        document.getElementById("marker-mode-badge").textContent = capitalize(mode) + " Mode";
        document.getElementById("marker-bar").style.width = pct + "%";
        document.getElementById("marker-pct").textContent = pct + "%";
        document.getElementById("marker-elapsed").textContent = "⏱ " + formatTime(elapsed);
        document.getElementById("marker-title").textContent = data.step || "";
        document.getElementById("marker-filename").textContent = data.current_file || "";
        document.getElementById("marker-eta").textContent = data.stage_eta_sec ? "ETA: ~" + data.stage_eta_sec + "s" : "ETA: --";

        // Update stages
        updateStages(data);

        // Update sub-bar
        const sn = data.stage_n || 0;
        const st = data.stage_total || 1;
        const subPct = Math.min(100, Math.round((sn / st) * 100));
        document.getElementById("stage-sub-bar").style.width = subPct + "%";
        document.getElementById("stage-sub-label").textContent = sn + "/" + st;

        // Per-file info
        if (data.per_file_done) {
          const list = document.getElementById("per-file-list");
          const fileName = data.current_file || "";
          const exists = Array.from(list.children).some(li => li.getAttribute("data-file") === fileName);
          if (!exists) {
            const li = document.createElement("li");
            li.setAttribute("data-file", fileName);
            li.className = "pf-done";
            li.innerHTML = "✅ " + fileName + " — done in " + data.per_file_done + "s";
            list.prepend(li);
          }
        }
      }

      if (view === "downloads") {
        document.getElementById("dl-mode-badge").textContent = capitalize(mode) + " Mode";
        document.getElementById("dl-bar").style.width = pct + "%";
        document.getElementById("dl-pct").textContent = pct + "%";
        document.getElementById("dl-elapsed").textContent = "Finished in " + formatTime(elapsed);
        document.getElementById("dl-status").textContent = data.step || "";

        // Show files if provided
        if (data.files && data.files.length > 0) {
          const list = document.getElementById("file-list");
          list.innerHTML = data.files.map(f =>
            `<li><span>${f.name} (${f.size})</span><a href="/api/download/${f.name}" download>Download</a></li>`
          ).join("");
        } else {
          refreshFiles();
        }
      }
    }

    function capitalize(s) { return s.charAt(0).toUpperCase() + s.slice(1); }

    function formatTime(sec) {
      if (sec < 60) return sec + "s";
      const m = Math.floor(sec / 60);
      const s = Math.floor(sec % 60);
      return m + ":" + String(s).padStart(2, "0");
    }

    function updateStages(data) {
      const currentStage = data.marker_stage || "";
      const pills = document.querySelectorAll("#stages-row .stage-pill");
      let foundCurrent = false;

      // "Complete" means all stages are done
      if (currentStage === "Complete") {
        pills.forEach(pill => {
          pill.className = "stage-pill done";
          pill.querySelector(".icon").textContent = "✅";
          pill.querySelector(".icon").className = "icon done";
        });
        return;
      }

      pills.forEach(pill => {
        const stage = pill.dataset.stage;
        const icon = pill.querySelector(".icon");

        if (stage === currentStage) {
          foundCurrent = true;
          pill.className = "stage-pill current";
          icon.textContent = "▶";
          icon.className = "icon current";
        } else if (!foundCurrent) {
          pill.className = "stage-pill done";
          icon.textContent = "✅";
          icon.className = "icon done";
        } else {
          pill.className = "stage-pill";
          icon.textContent = "○";
          icon.className = "icon pending";
        }
      });
    }

    function updateChecklist(listId, data) {
      const items = document.querySelectorAll("#" + listId + " li");
      items.forEach(li => {
        const label = li.querySelector(".label");
        const icon = li.querySelector(".icon");
        icon.className = "icon pending";
        icon.textContent = "○";
        label.className = "label";
      });

      // Find which items are done vs current
      // Simple heuristic: first item whose text matches a substring of the step is current
      const step = (data.step || "").toLowerCase();
      let foundCurrent = false;
      items.forEach(li => {
        const label = li.querySelector(".label");
        const icon = li.querySelector(".icon");
        const text = label.textContent.toLowerCase();

        if (!foundCurrent && step.includes(text)) {
          icon.textContent = "▶";
          icon.className = "icon current";
          label.className = "label current";
          foundCurrent = true;
        } else if (!foundCurrent) {
          icon.textContent = "✅";
          icon.className = "icon done";
          label.className = "label";
        } else {
          icon.textContent = "○";
          icon.className = "icon pending";
          label.className = "label";
        }
      });
    }

    // ============================================================
    // Settings: show/hide fields by mode
    // ============================================================
    function updateVisibility(mode) {
      const showDrive = mode === "single" || mode === "split";
      const showFolder = mode === "folder";
      const showSplit = mode === "split";
      document.getElementById("field-drive-link").style.display = showDrive ? "" : "none";
      document.getElementById("field-folder-link").style.display = showFolder ? "" : "none";
      document.getElementById("field-split-page").style.display = showSplit ? "" : "none";
    }

    document.querySelectorAll('input[name="mode"]').forEach(el => {
      el.addEventListener("change", () => updateVisibility(document.querySelector('input[name="mode"]:checked').value));
    });
    updateVisibility("single");

    // ============================================================
    // API calls: save / load / reset
    // ============================================================
    const API_SAVE  = "/api/save";
    const API_LOAD  = "/api/load";
    const API_RESET = "/api/reset";

    function getValues() {
      return {
        mode: document.querySelector('input[name="mode"]:checked').value,
        split_at_page: parseInt(document.getElementById("split_at_page").value) || 0,
        drive_link: document.getElementById("drive_link").value,
        drive_folder_link: document.getElementById("drive_folder_link").value,
        zip_name: document.getElementById("zip_name").value,
        add_page_markers: document.getElementById("add_page_markers").checked,
        github_token: document.getElementById("github_token").value,
      };
    }

    function setValues(v) {
      const radio = document.querySelector(`input[name="mode"][value="${v.mode}"]`);
      if (radio) radio.checked = true;
      document.getElementById("split_at_page").value = v.split_at_page;
      document.getElementById("drive_link").value = v.drive_link || "";
      document.getElementById("drive_folder_link").value = v.drive_folder_link || "";
      document.getElementById("zip_name").value = v.zip_name;
      document.getElementById("add_page_markers").checked = v.add_page_markers;
      document.getElementById("github_token").value = v.github_token || "";
    }

    function setStatus(msg, ok) {
      const el = document.getElementById("status");
      el.textContent = msg;
      el.className = ok ? "success" : "";
    }

    async function loadConfig() {
      try {
        const res = await fetch(API_LOAD);
        const data = await res.json();
        if (data && typeof data === "object") {
          setValues(data);
          updateVisibility(data.mode || "single");
        }
      } catch { /* defaults */ }
    }

    // Save
    document.getElementById("btn-save").addEventListener("click", async () => {
      const btn = document.getElementById("btn-save");
      btn.disabled = true; btn.textContent = "Saving...";
      try {
        const vals = getValues();
        const res = await fetch(API_SAVE, {
          method: "POST",
          headers: { "Content-Type": "application/json" },
          body: JSON.stringify(vals),
        });
        const result = await res.json();
        const msg = result.message || "Saved.";
        setStatus(msg, true);
      } catch (e) {
        setStatus("Error: " + e.message, false);
      } finally {
        btn.disabled = false; btn.textContent = "Save & Apply";
      }
    });

    document.getElementById("btn-reset").addEventListener("click", async () => {
      try {
        const res = await fetch(API_RESET, { method: "POST" });
        const data = await res.json();
        if (data && typeof data === "object") setValues(data);
        setStatus("Reset to defaults. Click Save & Apply to persist.", true);
      } catch (e) {
        setStatus("Error: " + e.message, false);
      }
    });

    loadConfig();

    // ============================================================
    // Downloads (used by view 4 and settings view)
    // ============================================================
    const API_FILES = "/api/files";
    const DOWNLOAD_BASE = "/api/download/";

    async function refreshFiles() {
      const list = document.getElementById("file-list");
      list.innerHTML = '<li class="no-files">Loading...</li>';
      try {
        const res = await fetch(API_FILES);
        const data = await res.json();
        if (!data.files || data.files.length === 0) {
          list.innerHTML = '<li class="no-files">No output files yet.</li>';
          return;
        }
        list.innerHTML = data.files.map(f =>
          `<li><span>${f.name} (${f.size})</span><a href="${DOWNLOAD_BASE}${f.name}" download>Download</a></li>`
        ).join("");
      } catch {
        list.innerHTML = '<li class="no-files">Could not fetch file list.</li>';
      }
    }

    document.getElementById("btn-refresh-files").addEventListener("click", refreshFiles);

    // ============================================================
    // Start polling on page load
    // ============================================================
    pollStatus();
    setInterval(pollStatus, 3000);
  </script>
</body>
</html>"""

# Start server in background thread
def run_server():
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=7862, log_level="warning")

thread = threading.Thread(target=run_server, daemon=True)
thread.start()
time.sleep(2)

print("=" * 60)
print("UI SERVER RUNNING")
print("=" * 60)

# Cloudflare Tunnel (no API key needed)
print("\nStarting Cloudflare Tunnel...")

import subprocess, re
import queue, threading as _t

public_url = None

def read_output(proc, out_q):
    for line in iter(proc.stdout.readline, ""):
        out_q.put(line)

proc = subprocess.Popen(
    [CLOUDFLARED_PATH, "tunnel", "--url", "http://localhost:7862"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)

q = queue.Queue()
reader = _t.Thread(target=read_output, args=(proc, q), daemon=True)
reader.start()

# Wait for tunnel URL (up to 30 seconds)
for _ in range(30):
    try:
        line = q.get(timeout=1)
        clean = line.strip()
        if clean:
            print(f"  {clean}")
        match = re.search(r'https://[a-zA-Z0-9\-]+\.trycloudflare\.com', line)
        if match:
            public_url = match.group(0)
            break
    except queue.Empty:
        pass

if public_url:
    print(f"\n🌍 Tunnel URL: {public_url}")
    print(f"\nOpen the URL above in your browser to configure settings.")
    print(f"Then return here and run the next cell.")
else:
    print(f"\n  Local: http://127.0.0.1:7862")
    print(f"  (cloudflared did not produce a tunnel URL)")

In [4]:
import json, os
import time as _proc_time
_PROCESS_START = _proc_time.time()
_STATUS_PATH = "/kaggle/working/status.json"

def write_status(view, progress, step, **kw):
    status = {"view": view, "progress": progress, "step": step, "elapsed": _proc_time.time() - _PROCESS_START}
    status["mode"] = config.get("mode", "single")
    status.update(kw)
    with open(_STATUS_PATH, "w") as f:
        json.dump(status, f)


# ============================================================
# CONFIG: Loaded from config.json (set via UI) or defaults
# ============================================================

CONFIG_PATH = "/kaggle/working/config.json"

DEFAULTS = {
    "mode": "single",
    "split_at_page": 12,
    "batch_input_dir": "/kaggle/working/pdf_batch",
    "drive_link": "",
    "drive_folder_link": "",
    "zip_name": "output",
    "add_page_markers": False,
    "github_token": "",
}

config = dict(DEFAULTS)
if os.path.exists(CONFIG_PATH):
    try:
        with open(CONFIG_PATH) as f:
            saved = json.load(f)
        config.update({k: v for k, v in saved.items() if k in DEFAULTS})
        print(f"Loaded config from {CONFIG_PATH}")
    except Exception as e:
        print(f"Failed to load config.json: {e}")

MODE               = config["mode"]
SPLIT_AT_PAGE      = config["split_at_page"]
BATCH_INPUT_DIR    = config["batch_input_dir"]
DRIVE_LINK         = config["drive_link"]
DRIVE_FOLDER_LINK  = config["drive_folder_link"]
ZIP_NAME           = config["zip_name"]
ADD_PAGE_MARKERS   = config["add_page_markers"]
GITHUB_TOKEN       = config["github_token"]

# Pass token to Cell 9 via env var
if GITHUB_TOKEN:
    os.environ["GITHUB_TOKEN"] = GITHUB_TOKEN

print()
print("=" * 60)
print("CURRENT CONFIG")
print("=" * 60)
print(f"  MODE              = {MODE}")
print(f"  SPLIT_AT_PAGE     = {SPLIT_AT_PAGE}")
print(f"  BATCH_INPUT_DIR   = {BATCH_INPUT_DIR}")
print(f"  DRIVE_LINK        = {'<set>' if DRIVE_LINK else '<empty>'}")
print(f"  DRIVE_FOLDER_LINK = {'<set>' if DRIVE_FOLDER_LINK else '<empty>'}")
print(f"  ZIP_NAME          = {ZIP_NAME}")
print(f"  ADD_PAGE_MARKERS  = {ADD_PAGE_MARKERS}")
print(f"  GITHUB_TOKEN      = {'<set>' if GITHUB_TOKEN else '<empty>'}")
print("=" * 60)
print()



⚠️ CRITICAL: Review the config cell below. Type 'continue' to proceed:  continue


✅ Validation successful. Proceeding with execution...


In [5]:
# Config moved to Cell 4. This cell kept for backward compatibility.

In [6]:
import urllib.request, os, re

print("\n" + "=" * 70)
print("STEP 4: DOWNLOAD PDF(s)")
print("=" * 70)

os.makedirs(BATCH_INPUT_DIR, exist_ok=True)

# --- SINGLE FILE ---
if MODE == "single" or MODE == "split":
    write_status("preparing", 0.10, "Downloading PDF from Drive...")
    match = re.search(r'/d/([a-zA-Z0-9_-]+)|id=([a-zA-Z0-9_-]+)', DRIVE_LINK)
    if not match:
        raise ValueError("Invalid Google Drive file link.")
    FILE_ID = match.group(1) if match.group(1) else match.group(2)
    PDF_URL = f"https://drive.google.com/uc?export=download&id={FILE_ID}"
    
    timestamp = __import__('datetime').datetime.now().strftime("%Y%m%d_%H%M%S")
    pdf_name = f"doc_{timestamp}.pdf"
    PDF_PATH = os.path.join(os.path.dirname(BATCH_INPUT_DIR), pdf_name)
    
    print(f"⬇️  Downloading single file...")
    urllib.request.urlretrieve(PDF_URL, PDF_PATH)
    
    if os.path.exists(PDF_PATH) and os.path.getsize(PDF_PATH) > 1000:
        print(f"✅ Saved: {os.path.basename(PDF_PATH)} ({os.path.getsize(PDF_PATH)/1024/1024:.2f} MB)")
        write_status("preparing", 0.25, f"Downloaded {os.path.basename(PDF_PATH)}. Starting Marker.")
    else:
        raise FileNotFoundError("PDF download failed. Check Drive sharing.")

# ---  FOLDER ---
elif DRIVE_FOLDER_LINK.strip():
    try:
        import gdown
    except ImportError:
        !pip install gdown -q
        import gdown

    write_status("preparing", 0.10, "Downloading PDFs from Drive folder...")
    print(f"📁 Downloading folder...")
    folder_match = re.search(r'/folders/([a-zA-Z0-9_-]+)', DRIVE_FOLDER_LINK)
    if not folder_match:
        raise ValueError("Invalid Google Drive folder link.")

    folder_id = folder_match.group(1)
    !gdown --folder "https://drive.google.com/drive/folders/{folder_id}" -O "{BATCH_INPUT_DIR}" --remaining-ok

    # Flatten: move any nested PDFs to root of BATCH_INPUT_DIR
    for root, dirs, files in os.walk(BATCH_INPUT_DIR):
        for f in files:
            if f.lower().endswith('.pdf'):
                src = os.path.join(root, f)
                dst = os.path.join(BATCH_INPUT_DIR, f)
                if src != dst:
                    import shutil
                    shutil.move(src, dst)
    # Remove empty subfolders
    for root, dirs, files in os.walk(BATCH_INPUT_DIR, topdown=False):
        for d in dirs:
            dpath = os.path.join(root, d)
            if not os.listdir(dpath):
                os.rmdir(dpath)

    pdf_files = sorted([f for f in os.listdir(BATCH_INPUT_DIR) if f.lower().endswith('.pdf')])
    print(f"✅ Downloaded {len(pdf_files)} PDF(s) to {BATCH_INPUT_DIR}")
    for pf in pdf_files:
        print(f"   {pf}")
    write_status("preparing", 0.25, f"Downloaded {len(pdf_files)} PDF(s). Starting Marker.")

else:
    raise ValueError("Set either DRIVE_LINK or DRIVE_FOLDER_LINK")


STEP 4: DOWNLOAD PDF(s)
📁 Downloading folder...
Retrieving folder contents
Processing file 1Fo72Kl0hgDAVbAKhmudTZK5aEFJTG7Jy DOC-20260326-WA0034.pdf
Processing file 1q4JZyxUYCSQvCh7466SyjcyTx3u7xzXW Jews and Muslims in the Maghreb (Version anglaise) .pdf
Processing file 1QHq0Tm0PXFVpZcsySEV4UbwMMXj_0IiO سيكولوجية الصحة كامل 2025-2026-1.pdf
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1Fo72Kl0hgDAVbAKhmudTZK5aEFJTG7Jy
To: /kaggle/working/pdf_batch/DOC-20260326-WA0034.pdf
100%|███████████████████████████████████████| 1.09M/1.09M [00:00<00:00, 110MB/s]
Downloading...
From: https://drive.google.com/uc?id=1q4JZyxUYCSQvCh7466SyjcyTx3u7xzXW
To: /kaggle/working/pdf_batch/Jews and Muslims in the Maghreb (Version anglaise) .pdf
100%|█████████████████████████████████████████| 444k/444k [00:00<00:00, 116MB/s]
Downloading...
From: https://drive.google.com/uc?id=1QHq0Tm0PXFVpZcsySEV4UbwMM

In [ ]:
#step 5
import time, os, glob, subprocess, math, shutil, datetime, shlex, re, json
from concurrent.futures import ThreadPoolExecutor
from tqdm.notebook import tqdm

print("\n" + "=" * 70)
print("STEP 5: RUNNING MARKER EXTRACTION")
print("=" * 70)


OUTPUT_DIR = os.path.join("/kaggle/working", "marker_output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

try:
    PDF_PATH
except NameError:
    existing = glob.glob("/kaggle/working/*.pdf")
    PDF_PATH = existing[0] if existing else None

import torch
gpu_count = torch.cuda.device_count()
print(f"🎮 GPUs: {gpu_count}")
print(f"⚙️  Mode: {MODE}\n")

start = time.time()
write_status("marker", 0.30, "Initializing Marker workers...")

# ---------------------------------------------------------------------
# PERSISTENT WORKER REGISTRY (Prevents model reloading across executions)
# ---------------------------------------------------------------------
if "_MARKER_WORKERS" not in globals():
    _MARKER_WORKERS = {}

def get_marker_worker(device):
    global _MARKER_WORKERS
    # Re-use worker if it's already active and resident in GPU VRAM
    if device in _MARKER_WORKERS:
        proc = _MARKER_WORKERS[device]
        if proc.poll() is None:
            return proc

    # Inline worker loop that keeps models warm in VRAM across calls
    worker_code = """import sys, os, json
import marker.models

# --- FIX: patch marker.models BEFORE importing convert_single ---
# convert_single (or something it imports) very likely does
# "from marker.models import create_model_dict" at its own top level.
# That statement copies a direct reference to the ORIGINAL function into
# convert_single's namespace at import time. If we patch marker.models
# AFTER that import already happened, convert_single's copy is left
# untouched and every job call reloads a brand-new model set on top of
# the one loaded during warmup -> VRAM grows every call -> CUDA OOM.
# Patching first means any "from marker.models import X" that happens
# during or after this point binds to OUR cached wrapper instead.

orig_create_model_dict = getattr(marker.models, 'create_model_dict', None)
orig_load_all_models = getattr(marker.models, 'load_all_models', None)

cached_models = None

def patched_create_model_dict(*args, **kwargs):
    global cached_models
    if cached_models is None:
        cached_models = orig_create_model_dict(*args, **kwargs)
    return cached_models

def patched_load_all_models(*args, **kwargs):
    global cached_models
    if cached_models is None:
        cached_models = orig_load_all_models(*args, **kwargs)
    return cached_models

if orig_create_model_dict:
    marker.models.create_model_dict = patched_create_model_dict
if orig_load_all_models:
    marker.models.load_all_models = patched_load_all_models

# NOW it's safe to import convert_single: it will see the patched functions.
try:
    from marker.scripts.convert_single import convert_single_cli as run_cli
except ImportError:
    from convert_single import main as run_cli

# Belt-and-suspenders: if convert_single (or a module it already imported
# earlier in the process) grabbed its own reference some other way, patch
# that module's namespace directly too, so it can't bypass the cache.
import sys as _sys
for _modname in ("marker.scripts.convert_single", "convert_single"):
    _mod = _sys.modules.get(_modname)
    if _mod is not None:
        if hasattr(_mod, "create_model_dict"):
            _mod.create_model_dict = patched_create_model_dict
        if hasattr(_mod, "load_all_models"):
            _mod.load_all_models = patched_load_all_models

print("🤖 Warm up: Caching marker models to GPU VRAM...", flush=True)
if orig_create_model_dict:
    try: patched_create_model_dict()
    except Exception: pass
if orig_load_all_models:
    try: patched_load_all_models()
    except Exception: pass

print("__WORKER_READY__", flush=True)

while True:
    line = sys.stdin.readline()
    if not line:
        break
    line = line.strip()
    if not line:
        continue
    try:
        data = json.loads(line)
        sys.argv = ["marker_single", data["pdf_path"], "--output_dir", data["out_dir"]]
        try:
            run_cli()
        except SystemExit as e:
            if e.code != 0:
                raise RuntimeError(f"CLI exited with code {e.code}")
        print("__WORKER_SUCCESS__", flush=True)
    except Exception as e:
        print(f"__WORKER_ERROR__: {str(e)}", flush=True)
"""
    worker_file = f"/kaggle/working/persistent_marker_worker_{device or 'default'}.py"
    with open(worker_file, "w", encoding="utf-8") as f:
        f.write(worker_code)

    env = os.environ.copy()
    if device is not None:
        env["CUDA_VISIBLE_DEVICES"] = str(device)

    print(f"🚀 Initializing persistent worker for GPU {device or 'default'} (Models will load ONCE)...")
    proc = subprocess.Popen(
        [sys.executable, "-u", worker_file],
        stdin=subprocess.PIPE,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env
    )

    # Keep track of warmup logs
    init_logs = []
    while True:
        line = proc.stdout.readline()
        if not line:
            print("".join(init_logs))
            raise RuntimeError(f"Worker for device {device} exited unexpectedly during warmup.")
        init_logs.append(line)
        cleaned = line.strip()
        if cleaned and not cleaned.startswith("__"):
            print(f"  [Worker GPU {device or 'default'}] {cleaned}")
        if "__WORKER_READY__" in line:
            break

    _MARKER_WORKERS[device] = proc
    return proc

# ---------------------------------------------------------------------
# PROGRESS-AWARE marker RUNNER (Leverages persistent VRAM workers)
# ---------------------------------------------------------------------
_PROGRESS_RE = re.compile(r'^(?P<desc>[^:]+):\s+(?P<pct>\d+)%\|.*?\|\s*(?P<n>\d+)/(?P<total>\d+)')
_LOADED_RE = re.compile(r'^Loaded (.+?) model')

def run_marker(pdf_path, out_dir, device=None, label=None, verbose_milestones=True):
    """Runs marker inside a long-lived process, showing a live tqdm bar instead of
    dumping raw output. On failure, prints the FULL captured log + raises."""
    name = label or os.path.basename(pdf_path)
    
    # Direct the file processing target to our active VRAM worker
    proc = get_marker_worker(device)
    
    # Pass execution targets securely to the worker via JSON stream
    job_data = json.dumps({"pdf_path": os.path.abspath(pdf_path), "out_dir": os.path.abspath(out_dir)})
    proc.stdin.write(job_data + "\n")
    proc.stdin.flush()

    log_lines = []
    seen_loaded = set()
    bar = None
    current_stage = None
    t0 = time.time()
    success = False

    try:
        while True:
            line = proc.stdout.readline()
            if not line:
                break
            log_lines.append(line)
            line = line.strip()
            if not line:
                continue

            if "__WORKER_SUCCESS__" in line:
                success = True
                break
            if "__WORKER_ERROR__" in line:
                print(f"\n❌ [{name}] worker processing FAILED — full log:\n")
                print("".join(log_lines))
                raise RuntimeError(f"{name}: marker processing failed: {line}")

            m = _LOADED_RE.match(line)
            if m and verbose_milestones and m.group(1) not in seen_loaded:
                seen_loaded.add(m.group(1))
                tqdm.write(f"  ✓ [{name}] loaded {m.group(1)} model")
                continue

            m = _PROGRESS_RE.match(line)
            if m:
                stage, n, total = m.group('desc'), int(m.group('n')), int(m.group('total'))
                if stage != current_stage:
                    if bar is not None:
                        bar.close()
                    bar = tqdm(total=total, desc=f"[{name}] {stage}", leave=False)
                    current_stage = stage
                bar.n = n
                bar.refresh()
                # Update progress status
                stage_short = stage.split("] ")[-1] if "] " in stage else stage
                elapsed_marker = time.time() - start
                write_status("marker", 0.30 + 0.55 * (1 - (total - n) / max(total, 1)),
                    f"{stage_short} ({n}/{total})",
                    current_file=os.path.basename(pdf_path),
                    marker_stage=stage_short,
                    stage_n=n, stage_total=total)
    finally:
        if bar is not None:
            bar.close()

    elapsed = time.time() - t0

    if not success:
        print(f"\n❌ [{name}] marker worker crashed or disconnected — full log:\n")
        print("".join(log_lines))
        raise RuntimeError(f"{name}: marker worker disconnected unexpectedly.")

    print(f"  ✅ [{name}] done in {elapsed:.1f}s")
    write_status("marker", 0.88, f"File complete. Done in {elapsed:.1f}s",
        current_file=os.path.basename(pdf_path), marker_stage="Complete",
        stage_n=1, stage_total=1, per_file_done=round(elapsed, 1))

# -------------------------------------------------------------------------
# MODE 1 & 2: DEFAULT / BATCH_MULTIPLIER
# -------------------------------------------------------------------------
if MODE in ("default", "batch_multiplier"):
    if not PDF_PATH:
        raise NameError("No PDF found. Run Cell 4 first.")
    print(f"📄 {os.path.basename(PDF_PATH)}")
    run_marker(PDF_PATH, OUTPUT_DIR, label=os.path.basename(PDF_PATH))

    pdf_base = os.path.splitext(os.path.basename(PDF_PATH))[0]
    output_path = os.path.join(OUTPUT_DIR, pdf_base)

# -------------------------------------------------------------------------
# MODE 3: SPLIT (2 GPUs parallel)
# -------------------------------------------------------------------------
elif MODE == "split":
    if not PDF_PATH:
        raise NameError("No PDF found. Run Cell 4 first.")

    if gpu_count < 2:
        print("⚠️  Only 1 GPU. Falling back to default.")
        run_marker(PDF_PATH, OUTPUT_DIR, label=os.path.basename(PDF_PATH))
        pdf_base = os.path.splitext(os.path.basename(PDF_PATH))[0]
        output_path = os.path.join(OUTPUT_DIR, pdf_base)
    else:
        try:
            from pypdf import PdfReader, PdfWriter
        except ImportError:
            !pip install pypdf -q
            from pypdf import PdfReader, PdfWriter

        reader = PdfReader(PDF_PATH)
        total_pages = len(reader.pages)
        split_page = SPLIT_AT_PAGE if (0 < SPLIT_AT_PAGE < total_pages) else math.ceil(total_pages / 2)

        print(f"✂️  Split at page {split_page} / {total_pages}")
        print(f"   Part 1: pages 1-{split_page}")
        print(f"   Part 2: pages {split_page+1}-{total_pages}")

        part1 = os.path.join("/kaggle/working", "split_part1.pdf")
        part2 = os.path.join("/kaggle/working", "split_part2.pdf")

        w1, w2 = PdfWriter(), PdfWriter()
        for i, page in enumerate(reader.pages):
            (w1 if i < split_page else w2).add_page(page)
        with open(part1, "wb") as f: w1.write(f)
        with open(part2, "wb") as f: w2.write(f)

        out1 = os.path.join(OUTPUT_DIR, "_raw_part1")
        out2 = os.path.join(OUTPUT_DIR, "_raw_part2")

        print("\n🚀 GPU 0 + GPU 1 in parallel...\n")
        with ThreadPoolExecutor(max_workers=2) as ex:
            f1 = ex.submit(run_marker, part1, out1, "0", "GPU0")
            f2 = ex.submit(run_marker, part2, out2, "1", "GPU1")
            f1.result(); f2.result()

        md1 = glob.glob(f"{out1}/**/*.md", recursive=True)
        md2 = glob.glob(f"{out2}/**/*.md", recursive=True)
        if not md1 or not md2:
            raise FileNotFoundError("Marker produced no markdown output. Check GPU errors above.")

        base_name = os.path.splitext(os.path.basename(PDF_PATH))[0]
        merged_dir = os.path.join(OUTPUT_DIR, base_name)
        os.makedirs(merged_dir, exist_ok=True)

        merged_md = os.path.join(merged_dir, f"{base_name}.md")
        with open(merged_md, "w", encoding="utf-8") as out_f:
            out_f.write(f"{'='*50}\n📄 PART 1 — PAGES 1-{split_page}\n{'='*50}\n\n")
            with open(md1[0], "r", encoding="utf-8") as f: out_f.write(f.read())
            out_f.write(f"\n\n{'='*50}\n📄 PART 2 — PAGES {split_page+1}-{total_pages}\n{'='*50}\n\n")
            with open(md2[0], "r", encoding="utf-8") as f: out_f.write(f.read())

        for src_dir in [out1, out2]:
            for f in glob.glob(f"{src_dir}/**/*", recursive=True):
                if os.path.isfile(f) and not f.endswith('.md'):
                    shutil.copy2(f, merged_dir)

        shutil.rmtree(out1, ignore_errors=True)
        shutil.rmtree(out2, ignore_errors=True)
        os.remove(part1); os.remove(part2)

        output_path = merged_dir
        print(f"\n📄 Merged: {output_path}")

# -------------------------------------------------------------------------
# MODE 4: FOLDER (distribute across 2 GPUs)
# -------------------------------------------------------------------------
elif MODE == "folder":
    pdf_files = sorted([os.path.join(BATCH_INPUT_DIR, f) for f in os.listdir(BATCH_INPUT_DIR) if f.lower().endswith('.pdf')])
    if not pdf_files:
        raise FileNotFoundError(f"No PDFs in {BATCH_INPUT_DIR}")

    print(f"📁 {len(pdf_files)} PDF(s)")

    container_name = f"batch_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}"
    container_dir = os.path.join(OUTPUT_DIR, container_name)
    os.makedirs(container_dir, exist_ok=True)
    print(f"📦 Container: {container_name}/")

    def process_files(files, device, label):
        for pdf in files:
            name = os.path.splitext(os.path.basename(pdf))[0]
            out = os.path.join(container_dir, name)
            run_marker(pdf, out, device=device, label=f"{label}-{name}")

    if gpu_count >= 2 and len(pdf_files) >= 2:
        mid = (len(pdf_files) + 1) // 2
        g1, g2 = pdf_files[:mid], pdf_files[mid:]

        print("\n🚀 GPU 0 + GPU 1 in parallel...\n")
        with ThreadPoolExecutor(max_workers=2) as ex:
            f1 = ex.submit(process_files, g1, "0", "GPU0")
            f2 = ex.submit(process_files, g2, "1", "GPU1")
            f1.result(); f2.result()

        print(f"\n✅ Both GPUs finished")
    else:
        process_files(pdf_files, None, "CPU/GPU0")

    output_path = container_dir

else:
    raise ValueError(f"Unknown MODE: {MODE}")

elapsed = time.time() - start
print(f"\n{'='*70}")
print(f"✅ DONE in {elapsed:.1f} seconds ({elapsed/60:.1f} min)")
print(f"{'='*70}")


STEP 5: RUNNING MARKER EXTRACTION
🎮 GPUs: 2
⚙️  Mode: folder

📁 3 PDF(s)
📦 Container: batch_20260720_170530/

🚀 GPU 0 + GPU 1 in parallel...

🚀 Initializing persistent worker for GPU 0 (Models will load ONCE)...
🚀 Initializing persistent worker for GPU 1 (Models will load ONCE)...
  [Worker GPU 1] /usr/local/lib/python3.12/dist-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  [Worker GPU 1] warnings.warn(
  [Worker GPU 0] /usr/local/lib/python3.12/dist-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  [Worker GPU 0] warnings.warn(
  [Worker GPU 1] 🤖 Warm up: Caching marker models to GPU VRAM...
  [Worker GPU 0] 🤖 Warm up: Caching marker models to GPU VRAM...


[GPU0-DOC-20260326-WA0034] Recognizing Layout:   0%|          | 0/14 [00:00<?, ?it/s]

[GPU1-سيكولوجية الصحة كامل 2025-2026-1] Recognizing Layout:   0%|          | 0/22 [00:00<?, ?it/s]

[GPU0-DOC-20260326-WA0034] Running OCR Error Detection:   0%|          | 0/1 [00:00<?, ?it/s]

[GPU0-DOC-20260326-WA0034] Detecting bboxes:   0%|          | 0/1 [00:00<?, ?it/s]

...

[GPU0-DOC-20260326-WA0034] Recognizing Text:   0%|          | 0/142 [00:00<?, ?it/s]

  ✅ [GPU0-DOC-20260326-WA0034] done in 60.7s


[GPU0-Jews and Muslims in the Maghreb (Version anglaise) ] Recognizing Layout:   0%|          | 0/15 [00:00<?,…

[GPU0-Jews and Muslims in the Maghreb (Version anglaise) ] Running OCR Error Detection:   0%|          | 0/2 […

  ✅ [GPU0-Jews and Muslims in the Maghreb (Version anglaise) ] done in 12.3s


In [ ]:
import glob, os, shutil, re, datetime

print("\n" + "=" * 70)
print("STEP 6: FINALIZE OUTPUT")
print("=" * 70)



OUTPUT_DIR = os.path.join("/kaggle/working", "marker_output")
WORKING_DIR = "/kaggle/working"

# Recover output_path from Cell 5, or auto-detect
try:
    output_path
except NameError:
    subfolders = [f.path for f in os.scandir(OUTPUT_DIR) if f.is_dir()]
    if not subfolders:
        print("❌ No output found.")
        raise SystemExit
    output_path = max(subfolders, key=os.path.getctime)
    print(f"⚠️  Auto-detected: {output_path}")

md_files = glob.glob(f"{output_path}/*.md")
if not md_files:
    md_files = glob.glob(f"{output_path}/**/*.md", recursive=True)

# 1. Optional page markers
if ADD_PAGE_MARKERS and md_files:
    for md_path in md_files:
        with open(md_path, "r", encoding="utf-8") as f:
            content = f.read()
        page_pattern = r'!\[(.*?)\]\((_page_(\d+)_[^)]+)\)'
        seen = set()
        new_content = content
        for m in reversed(list(re.finditer(page_pattern, content))):
            p = int(m.group(3))
            if p not in seen:
                seen.add(p)
                marker = f"\n\n{'='*50}\n📄 PAGE {p + 1}\n{'='*50}\n\n"
                new_content = new_content[:m.start()] + marker + new_content[m.start():]
        if 0 not in seen and new_content.strip():
            new_content = f"{'='*50}\n📄 PAGE 1\n{'='*50}\n\n" + new_content
        with open(md_path, "w", encoding="utf-8") as f:
            f.write(new_content)
    print(f"✅ Page markers added to {len(md_files)} file(s)")

# 2. Determine zip name
if ZIP_NAME.strip():
    zip_name = ZIP_NAME.strip()
    if not zip_name.endswith(".zip"):
        zip_name += ".zip"
else:
    zip_name = os.path.basename(output_path) + ".zip"

zip_path = os.path.join(WORKING_DIR, zip_name)

# 3. Zip
write_status("downloads", 0.93, f"Zipping {zip_name}...")
print(f"\n📦 Zipping: {os.path.basename(output_path)} → {zip_name}")
shutil.make_archive(zip_path.replace(".zip", ""), 'zip', output_path)
print(f"✅ {zip_path} ({os.path.getsize(zip_path)/1024/1024:.1f} MB)")
write_status("downloads", 1.0, "Complete! Files ready for download.")

# 4. Clean loose folder
print(f"\n🧹 Cleaning up...")
shutil.rmtree(output_path)
print(f"   Deleted: {output_path}")

print(f"\n{'='*70}")
print(f"📥 FINAL OUTPUT: {zip_path}")
print(f"{'='*70}")

In [ ]:
##GITHUB_UPLOAD
import os, glob, subprocess, shutil, json
from IPython.display import display, HTML

print("\n" + "=" * 70)
print("STEP 7: PUSH TO GITHUB")
print("=" * 70)

# ==================== CONFIG ====================
GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "") or (json.load(open("/kaggle/working/config.json"))["github_token"] if os.path.exists("/kaggle/working/config.json") else "")  # Your Personal Access Token
GITHUB_USER = "freegamefire03-boop"       # e.g., "bahaeeddine"
REPO_NAME = "kaggle_marker_output"           # e.g., "pdf-extracts"
BRANCH = "main"                            # or "master"
COMMIT_MSG = "default"
# =================================================

REPO_URL = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git"
TEMP_DIR = "/tmp/github_push"
WORKING_DIR = "/kaggle/working"

# 1. Find ONLY zip files in /kaggle/working root (excludes all subfolders)
zip_files = sorted(glob.glob(os.path.join(WORKING_DIR, "*.zip")))

if not zip_files:
    print("❌ No zip files found in /kaggle/working/")
    print("   Run Cell 6 first to generate output zips.")
    raise SystemExit

print(f"📦 Found {len(zip_files)} zip(s) to push:")
for z in zip_files:
    size = os.path.getsize(z) / 1024 / 1024
    print(f"   {os.path.basename(z)} ({size:.1f} MB)")

# 2. Clone repo into temp folder
if os.path.exists(TEMP_DIR):
    shutil.rmtree(TEMP_DIR)

write_status("downloads", 0.95, "Pushing to GitHub...")
print(f"\n⬇️  Cloning {REPO_NAME}...")
result = subprocess.run(["git", "clone", REPO_URL, TEMP_DIR], capture_output=True, text=True)
if result.returncode != 0 and "empty repository" not in result.stderr.lower():
    print(f"⚠️  Clone issue: {result.stderr}")
    print("   Attempting fresh init...")
    os.makedirs(TEMP_DIR, exist_ok=True)
    subprocess.run(["git", "init"], cwd=TEMP_DIR, check=True)
    subprocess.run(["git", "remote", "add", "origin", REPO_URL], cwd=TEMP_DIR, check=True)
    subprocess.run(["git", "checkout", "-b", BRANCH], cwd=TEMP_DIR, check=True)
else:
    print(f"✅ Cloned to {TEMP_DIR}")

# 3. Copy ONLY zip files (no other folders leak in)
print(f"\n📤 Copying zips to repo...")
for z in zip_files:
    shutil.copy2(z, TEMP_DIR)
    print(f"   + {os.path.basename(z)}")

# 4. Git commit & push
print(f"\n🚀 Pushing to {BRANCH}...")
subprocess.run(["git", "config", "user.email", "kaggle@bot.com"], cwd=TEMP_DIR, check=True)
subprocess.run(["git", "config", "user.name", "Kaggle Bot"], cwd=TEMP_DIR, check=True)
subprocess.run(["git", "add", "*.zip"], cwd=TEMP_DIR, check=True)

# Track if the process finishes successfully to display links
show_links = False

# Check if there's anything to commit
status = subprocess.run(["git", "status", "--porcelain"], cwd=TEMP_DIR, capture_output=True, text=True)
if not status.stdout.strip():
    print("⚠️  Nothing new to commit (zips already exist in repo).")
    show_links = True
else:
    subprocess.run(["git", "commit", "-m", COMMIT_MSG], cwd=TEMP_DIR, check=True)
    push_result = subprocess.run(["git", "push", "origin", BRANCH], cwd=TEMP_DIR, capture_output=True, text=True)
    if push_result.returncode == 0:
        print(f"\n{'='*70}")
        print(f"✅ SUCCESSFULLY PUSHED to github.com/{GITHUB_USER}/{REPO_NAME}")
        print(f"{'='*70}")
        show_links = True
    else:
        print(f"❌ Push failed: {push_result.stderr}")

# --- GENERATE AND DISPLAY CLICKABLE HTML LINKS ---
if show_links:
    repo_url = f"https://github.com/{GITHUB_USER}/{REPO_NAME}"
    
    # Generate download layout for all detected zip files
    download_links_html = ""
    for z in zip_files:
        filename = os.path.basename(z)
        # Using raw.githubusercontent.com forces the browser to download the file directly
        direct_download_url = f"https://raw.githubusercontent.com/{GITHUB_USER}/{REPO_NAME}/{BRANCH}/{filename}"
        
        download_links_html += f"""
        <p style="margin: 10px 0;">
            <strong>💾 Direct Download ({filename}):</strong> <br>
            <a href="{direct_download_url}" target="_blank" style="color: #fff; background-color: #24292e; padding: 6px 12px; border-radius: 4px; text-decoration: none; font-size: 13px; font-weight: bold; display: inline-block; margin-top: 4px; border: 1px solid #1b1f23;">
                📥 Click to Download File
            </a>
        </p>
        """
        
    html_output = f"""
    <div style="padding: 15px; border: 1px solid #d1d5da; border-radius: 6px; background-color: #212529; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Helvetica, Arial, sans-serif; margin-top: 20px;">
        <h3 style="color: #28a745; margin-top: 0; display: flex; align-items: center;">🚀 File Links Generated Successfully!</h3>
        
        <p style="margin: 10px 0;">
            <strong>📊 GitHub Repository:</strong> <br>
            <a href="{repo_url}" target="_blank" style="color: #0366d6; text-decoration: none; font-weight: bold; font-size: 14px;">
                🔗 View Repository UI
            </a>
        </p>
        
        <hr style="border: 0; border-top: 1px solid #e1e4e8; margin: 15px 0;">
        {download_links_html}
    </div>
    """
    display(HTML(html_output))

In [ ]:
# import os
# import json
# import subprocess

# print("=" * 70)
# print("SAVING MODELS TO PERMANENT KAGGLE DATASET")
# print("=" * 70)

# # Your Kaggle username (change this!)
# YOUR_USERNAME = "bahaeeddineessadki"  # <-- EDIT THIS

# CACHE_DIR = "/kaggle/working/marker_models_cache"
# DATASET_SLUG = "marker-model-cache"
# DATASET_ID = f"{YOUR_USERNAME}/{DATASET_SLUG}"

# # 1. Create metadata file
# meta = {
#     "title": "Marker PDF Model Cache",
#     "id": DATASET_ID,
#     "licenses": [{"name": "CC0-1.0"}]
# }

# meta_path = os.path.join(CACHE_DIR, "dataset-metadata.json")
# with open(meta_path, "w") as f:
#     json.dump(meta, f, indent=2)

# print(f"✅ Metadata created: {meta_path}")

# # 2. Create/update the dataset
# print(f"\n⬆️  Uploading ~3GB to Kaggle Dataset: {DATASET_ID}")
# print("   This takes 3-5 minutes. Do not interrupt.\n")

# result = subprocess.run(
#     ["kaggle", "datasets", "create", "-p", CACHE_DIR, "--dir-mode", "zip", "--public"],
#     capture_output=True, text=True
# )

# print(result.stdout)
# if result.stderr:
#     print("Stderr:", result.stderr)

# print(f"\n{'='*70}")
# print(f"✅ DATASET CREATED: {DATASET_ID}")
# print(f"{'='*70}")
# print("\n🔗 URL: https://www.kaggle.com/datasets/" + DATASET_ID)

In [ ]:
import os, signal, subprocess, time, gc

print("🧹 Freeing GPU VRAM from orphaned marker workers...")

# 1. Kill any workers tracked in this kernel's registry
killed = 0
if "_MARKER_WORKERS" in globals():
    for device, proc in list(_MARKER_WORKERS.items()):
        if proc.poll() is None:
            proc.terminate()
            try:
                proc.wait(timeout=5)
            except subprocess.TimeoutExpired:
                proc.kill()
                proc.wait()
            killed += 1
            print(f"   killed tracked worker for device {device} (pid {proc.pid})")
    _MARKER_WORKERS.clear()

# 2. Catch any worker processes not tracked in this kernel session
#    (e.g. left over from a kernel restart / previous execution)
result = subprocess.run(["pgrep", "-f", "persistent_marker_worker"], capture_output=True, text=True)
stray_pids = [int(p) for p in result.stdout.split()]
for pid in stray_pids:
    try:
        os.kill(pid, signal.SIGKILL)
        killed += 1
        print(f"   killed stray worker pid {pid}")
    except ProcessLookupError:
        pass

if killed:
    time.sleep(2)  # give the driver a moment to reclaim memory after process exit

# 3. Clear any CUDA memory held directly by this notebook process, if any
try:
    import torch
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    print("   cleared this process's own CUDA cache")
except Exception as e:
    print(f"   (torch cleanup skipped: {e})")

print(f"\n✅ Done. Killed {killed} worker process(es).\n")

# 4. Show current VRAM state to confirm
subprocess.run(["nvidia-smi", "--query-gpu=index,memory.used,memory.total", "--format=csv"])